In [0]:
from pyspark.sql import SparkSession, DataFrame
from datetime import datetime
from pyspark.sql.functions import to_timestamp, col, date_trunc, current_timestamp, to_date, concat_ws, sha2
import logging

In [0]:
def date_now() -> datetime:
    return datetime.strptime(datetime.now().strftime("%Y-%m-%d %H:%M:%S"), "%Y-%m-%d %H:%M:%S")

In [0]:
spark = SparkSession.builder \
    .appName("Ingestion from csv to delta") \
    .getOrCreate()

logging.basicConfig(
    level=logging.INFO
    , format="%(asctime)s | %(levelname)s | %(message)s"
    , datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)

In [0]:
def columns_control(df_stage:DataFrame, primary_keys: str) -> DataFrame:
    primary_keys_list = primary_keys.split(",")
    df_with_hash_column = df_stage.withColumn("hash_column", sha2(concat_ws("||", *df_stage.columns), 256))
    df_with_hash_key = df_with_hash_column.withColumn("hash_key", sha2(concat_ws("||", *[col(key) for key in primary_keys_list]), 256))
    df_with_updated_at = df_with_hash_key.withColumn("updated_at", to_date(date_trunc("day", current_timestamp())))

    return df_with_updated_at

def read_csv(csv_name: str, keys: str) -> DataFrame: 
    df_stage = spark.read.csv(f'/Volumes/comercial-dev/source/csvs/{csv_name}.csv', header=True, inferSchema=True)
    return columns_control(df_stage, keys)


In [0]:
dbutils.widgets.text("csv_name", "", "CSV name")
dbutils.widgets.text("stage_table_name", "", "Stage name")
dbutils.widgets.dropdown("ambiente", "dev", ["dev", "prod"])
dbutils.widgets.text("keys", "", "keys")

csv_name = dbutils.widgets.get("csv_name")
stage_name = dbutils.widgets.get("stage_table_name")
ambiente = dbutils.widgets.get("ambiente")
keys = dbutils.widgets.get("keys")

In [0]:
logs_stage = {
    "dat_Hr_Process_Start": date_now(),
    "dat_Hr_Process_End":  '1900-01-01 00:00:00',
    "layer": "stage",
    "catalog": f'comercial-{ambiente}'.upper(),
    "source_Name": csv_name.upper(),
    "target_Name": stage_name.upper(),
    "status": None,
    "count_Rows": None,
}

In [0]:
def create_or_replace_stage_table(stage):
    stage.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(f'`comercial-{ambiente}`.`stage`.`{stage_name}`')

def save_logs(logs):
    logs.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(f'`comercial-{ambiente}`.`corporate`.`logs`')
        
  

In [0]:

try:
    logger.info("===========Start Process===========")
    logger.info("Start reading csv")
    df_stage = read_csv(csv_name, keys)
    logger.info("End reading csv")


    logger.info("Start Ingestion Stage")
    create_or_replace_stage_table(df_stage)
    logger.info("End Ingestion Stage")
    

    logger.info("Start save logs")

    logs_stage["status"] = "SUCCESS"
    logs_stage["count_Rows"] = df_stage.count()
    logs_stage["dat_Hr_Process_End"] = date_now()

    df_logs = spark.createDataFrame([logs_stage])
    
    save_logs(df_logs)
    logger.info("End saving logs")
    logger.info("===========End Process===========")
except Exception as e:
    logger.error(f'{e}')

    
